In [1]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_core.documents import Document

C:\Users\hardi_cezwich\AppData\Local\Temp\ipykernel_23204\4241866633.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models.")
]


In [3]:
embedding_model=HuggingFaceEmbeddings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
vector_store=Chroma.from_documents(documents=documents,embedding=embedding_model, collection_name="my_collection")

In [5]:
retriever=vector_store.as_retriever(search_kwargs={"k":2})

In [6]:
query="What is chroma used for?"
result=retriever.invoke(query)
result

[Document(metadata={}, page_content='Chroma is a vector database optimized for LLM-based search.'),
 Document(metadata={}, page_content='Embeddings convert text into high-dimensional vectors.')]

In [7]:
for i, docs in enumerate(result):
    print(f"\n--- Result {i+1} ---")
    print(docs.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


In [8]:
results=vector_store.similarity_search(query, k=2)

In [9]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Embeddings convert text into high-dimensional vectors.


**MMR Retriever**

In [10]:
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]


In [11]:
from langchain_community.vectorstores import FAISS

In [12]:
vectorstore=FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [13]:
retriever1=vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={"k":3, "lambda_mult": 1}
)

In [14]:
query1="What is langchain"
results=retriever1.invoke(query1)

In [15]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 3 ---
LangChain makes it easy to work with LLMs.


**Multi query retriever**

In [16]:
from langchain_classic.retrievers import MultiQueryRetriever

In [17]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [18]:
vectorstore1=FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [19]:
similarity_retriever=vectorstore1.as_retriever(search_type='similarity', search_kwargs={"k":5})

In [20]:
multi_query_retriever=MultiQueryRetriever.from_llm(
    retriever=vectorstore1.as_retriever(search_kwargs={"k":5}),
    llm=ChatMistralAI(model='mistral-small-2603')
)

In [21]:
query2="how to improve energy level and maintain balance?"

In [22]:
similarity_result=similarity_retriever.invoke(query2)
multiquery_result=multi_query_retriever.invoke(query2)

In [24]:
for i, doc in enumerate(similarity_result):
    print(f"\n --- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)




 --- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

 --- Result 2 ---
Regular walking boosts heart health and can reduce symptoms of depression.

 --- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

 --- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

 --- Result 5 ---
The solar energy system in modern homes helps balance electricity demand.
******************************************************************************************************************************************************


In [25]:
for i, doc in enumerate(multiquery_result):
    print(f"\n --- Result {i+1} ---")
    print(doc.page_content)


 --- Result 1 ---
The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.

 --- Result 2 ---
Black holes bend spacetime and store immense gravitational energy.

 --- Result 3 ---
Python balances readability with power, making it a popular system design language.

 --- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

 --- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.

 --- Result 6 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

 --- Result 7 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

 --- Result 8 ---
Photosynthesis enables plants to produce energy by converting sunlight.

 --- Result 9 ---
Deep sleep is crucial for cellular repair and emotional regulation.
